In [ ]:
# Code Block 1: Notebook description
#Notebook description
# This notebook evaluates the performance of a portfolio of assets on a buy-and-hold basis.
# It focuses on how multiple assets interact within the portfolio, rather than assessing a specific mechanical trading strategy.
# buy and hold is a good benchmark for any trading strategy, so it is important to evaluate the performance of a portfolio independently of
# trading strategies we may want to implement on the individual assets.


In [ ]:
# Code Block 2: Load Libraries
# Load Libraries
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from IPython.display import display
from schwab.auth import easy_client
import os
import sys
from pathlib import Path
# Seed the local package import when the notebook starts in a subfolder.
for _project_root_candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_project_root_candidate / "Quantapp" / "project.py").exists():
        if str(_project_root_candidate) not in sys.path:
            sys.path.insert(0, str(_project_root_candidate))
        break
else:
    raise RuntimeError("Could not locate the project root containing Quantapp.")

from Quantapp.project import ensure_project_root_on_path

PROJECT_ROOT = ensure_project_root_on_path()


from Quantapp.data import yf as qa_yf
from Quantapp.data import get_schwab_portfolio_snapshot
from Quantapp.data.adapters import SCHWAB_OPTION_SYMBOL_PATTERN

#from Quantapp.analytics import TimeSeriesAnalytics as Rolling
from Quantapp.data import MacroDataClient
from Quantapp.secrets import load_project_env, require_secret

load_project_env()

#qc = Rolling()
qe = MacroDataClient()

In [ ]:
# Code Block 3: Define functions and classes
#Define functions & Classes
#takes a dict of portfolio and their total amounts and directional (short or long value), converts dict to weightings instead of absolute values
def create_weighted_portfolio(portfolio):
    total = sum(abs(amount) for amount in portfolio.values())
    return {ticker: (amount / total) * (1 if direction == 'long' else -1)
            for (ticker, amount), direction in zip(portfolio.items(), ['long' if amount >= 0 else 'short' for amount in portfolio.values()])
    }
def create_weight_dict(portfolio):
    total = sum(abs(amount) for amount in portfolio.values())
    return {ticker: amount / total for ticker, amount in portfolio.items()}

def create_equal_weighted_dict(tickers):
    n = len(tickers)

    if n == 0:
        return {}

    equal_weight = 1 / n
    return {ticker: equal_weight for ticker in tickers}

def normalize_yf_ticker(ticker):
    if not isinstance(ticker, str):
        return ticker

    return ticker.strip().replace('/', '-')

def build_yf_ticker_map(tickers):
    return {ticker: normalize_yf_ticker(ticker) for ticker in tickers}

def zscore_series(series):
    mean = series.mean()
    std = series.std(ddof=0)

    if std == 0 or np.isnan(std):
        return pd.Series(0.0, index=series.index)

    return (series - mean) / std

def z_score(series):
    mean = series.mean()
    std = series.std(ddof=0)

    if std == 0 or np.isnan(std):
        return pd.Series(0.0, index=series.index)

    z = (series - mean) / std
    return z.replace([np.inf, -np.inf], np.nan)

def sharpe_annualized(series):
    mean = series.mean()
    std = series.std(ddof=0)

    if std == 0 or np.isnan(std):
        return 0.0

    return (mean / std) * np.sqrt(252)


In [ ]:
# Code Block 4: Define parameters
#Define parameters
time_frame_week = 7
time_frame_short = 21
time_frame_mid = 50
time_frame_long = 200
selected_time_frame = time_frame_long
CLIENT_ID = require_secret("SCHWAB_CLIENT_ID")
APP_SECRET = require_secret("SCHWAB_APP_SECRET")
CALLBACK_URL = os.getenv("SCHWAB_CALLBACK_URL", "https://127.0.0.1:8182")

_token_path = os.getenv("SCHWAB_TOKEN_PATH")
TOKEN_PATH = Path(_token_path).expanduser() if _token_path else PROJECT_ROOT / "schwab_token.json"
if not TOKEN_PATH.is_absolute():
    TOKEN_PATH = PROJECT_ROOT / TOKEN_PATH
TOKEN_PATH.parent.mkdir(parents=True, exist_ok=True)

#callback
period = '20y'
interval = '1d'
benchmark_str = 'SPY'


In [ ]:
# Code Block 5: Login to Schwab client
from schwab.auth import easy_client

client = easy_client(
    api_key=CLIENT_ID,
    app_secret=APP_SECRET,
    callback_url=CALLBACK_URL,
    token_path=str(TOKEN_PATH),
)

print(f"Schwab client ready. Token cache: {TOKEN_PATH}")


In [ ]:
# Code Block 6: Retrieve account and market data
# Schwab account retrieval and position normalization live in Quantapp.data.
portfolio_snapshot = get_schwab_portfolio_snapshot(client)

account_information = portfolio_snapshot.account_information
acct_map = portfolio_snapshot.account_numbers
acct_hash = portfolio_snapshot.account_hash
acct = portfolio_snapshot.account
positions = portfolio_snapshot.raw_positions
positions_df = portfolio_snapshot.option_positions
option_sentiment = portfolio_snapshot.option_sentiment
net_direction = portfolio_snapshot.net_direction
organized_positions = portfolio_snapshot.organized_positions
invested_symbols = portfolio_snapshot.invested_symbols
net_invested_amounts = portfolio_snapshot.net_invested_amounts
total_margin = portfolio_snapshot.total_margin
option_pattern = SCHWAB_OPTION_SYMBOL_PATTERN

# Retrieve core market data for benchmark and portfolio.
benchmark_symbol = benchmark_str if 'benchmark_str' in globals() else 'SPY'
benchmark_data = qa_yf.Ticker(benchmark_symbol).history(period=period, interval=interval)
invested_symbol_map = build_yf_ticker_map(invested_symbols)

if invested_symbol_map:
    portfolio_data = qa_yf.download(
        tickers=list(invested_symbol_map.values()),
        period=period,
        interval=interval,
        auto_adjust=True,
        threads=True,
        progress=False,
    )
    portfolio_closing_prices = portfolio_data['Close']
    if isinstance(portfolio_closing_prices, pd.Series):
        portfolio_closing_prices = portfolio_closing_prices.to_frame(name=invested_symbols[0])
    else:
        portfolio_closing_prices = portfolio_closing_prices.rename(
            columns={yf_ticker: ticker for ticker, yf_ticker in invested_symbol_map.items()}
        )
else:
    portfolio_closing_prices = pd.DataFrame(index=benchmark_data.index)

benchmark_close = benchmark_data['Close']
portfolio_closing_prices.index = portfolio_closing_prices.index.tz_localize(None)
benchmark_close.index = benchmark_close.index.tz_localize(None)

net_direction
raw_prices = portfolio_closing_prices.copy()


In [ ]:
# Code Block 7: Option P/L at expiration and net cost basis
# Retrieve net cost basis for each option grouped by ticker
# net cost basis = net_quantity * average_price * 100 (per contract)
from Quantapp.visualization.views.portfolio_profile.performance_structure import display_option_expiration_pl_view

if not positions_df.empty:
    positions_df['net_cost_basis'] = positions_df['net_quantity'] * positions_df['average_price'] * 100
    net_cost_basis = positions_df.groupby('underlying')['net_cost_basis'].sum().rename('net_cost_basis')
    display(net_cost_basis.to_frame())
else:
    net_cost_basis = pd.Series(dtype=float, name='net_cost_basis')
    print("No options positions found.")

available_option_underlyings = (
    sorted(positions_df['underlying'].dropna().unique().tolist())
    if not positions_df.empty else []
)
benchmark_label_for_beta = benchmark_str if 'benchmark_str' in globals() else 'SPY'
portfolio_latest_prices = (
    portfolio_closing_prices.ffill().iloc[-1].dropna().to_dict()
    if isinstance(portfolio_closing_prices, pd.DataFrame) and not portfolio_closing_prices.empty
    else {}
)
benchmark_current_price = (
    float(benchmark_close.ffill().iloc[-1])
    if isinstance(benchmark_close, pd.Series) and not benchmark_close.dropna().empty
    else np.nan
)
benchmark_returns_for_beta = (
    benchmark_close.pct_change().dropna()
    if isinstance(benchmark_close, pd.Series)
    else pd.Series(dtype=float)
)
asset_returns_for_beta = (
    portfolio_closing_prices.pct_change()
    if isinstance(portfolio_closing_prices, pd.DataFrame) and not portfolio_closing_prices.empty
    else pd.DataFrame()
)
underlying_beta_map = {}

if not asset_returns_for_beta.empty and not benchmark_returns_for_beta.empty:
    for underlying_name in asset_returns_for_beta.columns:
        aligned_returns = pd.concat(
            [
                asset_returns_for_beta[underlying_name].rename('asset'),
                benchmark_returns_for_beta.rename('benchmark'),
            ],
            axis=1,
        ).dropna()

        if len(aligned_returns) < 2 or aligned_returns['benchmark'].var() == 0:
            underlying_beta_map[underlying_name] = 1.0
            continue

        beta_value = aligned_returns['asset'].cov(aligned_returns['benchmark']) / aligned_returns['benchmark'].var()

        if pd.notna(beta_value) and np.isfinite(beta_value):
            underlying_beta_map[underlying_name] = float(beta_value)
        else:
            underlying_beta_map[underlying_name] = 1.0
else:
    underlying_beta_map = {underlying_name: 1.0 for underlying_name in available_option_underlyings}

if available_option_underlyings:
    option_expiration_pl_view = display_option_expiration_pl_view(
        positions_df,
        net_cost_basis=net_cost_basis,
        portfolio_latest_prices=portfolio_latest_prices,
        benchmark_current_price=benchmark_current_price,
        underlying_beta_map=underlying_beta_map,
        benchmark_label=benchmark_label_for_beta,
    )


In [ ]:
# Code Block 8: Option leg coverage audit
# =========================
# 8) Verify every raw Schwab option leg is represented in positions_df
# =========================
raw_option_rows = []
regex_miss_rows = []

for position in positions:
    instrument = position.get('instrument', {})

    if instrument.get('assetType') != 'OPTION':
        continue

    raw_symbol = instrument.get('symbol', '')
    normalized_symbol = ''.join((raw_symbol or '').split())
    match = option_pattern.match(normalized_symbol)
    expiration = pd.NaT
    parsed_option_type = pd.NA
    parsed_strike = pd.NA

    if match:
        expiration = pd.to_datetime('20' + match.group('expiration'), format='%Y%m%d', errors='coerce')
        parsed_option_type = match.group('option_type')
        parsed_strike = int(match.group('strike')) / 1000

    row = {
        'raw_symbol': raw_symbol,
        'normalized_symbol': normalized_symbol,
        'underlying_symbol': instrument.get('underlyingSymbol'),
        'put_call': instrument.get('putCall'),
        'expiration': expiration,
        'parsed_option_type': parsed_option_type,
        'parsed_strike': parsed_strike,
        'long_quantity': position.get('longQuantity', 0.0),
        'short_quantity': position.get('shortQuantity', 0.0),
        'average_price': position.get('averagePrice', 0.0),
    }
    raw_option_rows.append(row)

    if not match:
        regex_miss_rows.append(dict(row, drop_reason='symbol_did_not_match_option_pattern'))

raw_option_df = pd.DataFrame(raw_option_rows)
parsed_option_df = positions_df.copy()

if raw_option_df.empty:
    print('[Code Block 7] No raw option legs returned by Schwab for the selected account.')
else:
    raw_audit = (
        raw_option_df.groupby('normalized_symbol', dropna=False, as_index=False)
        .agg(
            raw_rows=('normalized_symbol', 'size'),
            raw_long_quantity=('long_quantity', 'sum'),
            raw_short_quantity=('short_quantity', 'sum'),
            raw_average_price=('average_price', 'first'),
            underlying_symbol=('underlying_symbol', 'first'),
            expiration=('expiration', 'first'),
            put_call=('put_call', 'first'),
        )
    )
    if parsed_option_df.empty:
        parsed_audit = pd.DataFrame(columns=[
            'symbol',
            'parsed_rows',
            'parsed_long_quantity',
            'parsed_short_quantity',
            'parsed_average_price',
        ])
    else:
        parsed_option_df['symbol'] = parsed_option_df['symbol'].astype(str)
        parsed_audit = (
            parsed_option_df.groupby('symbol', as_index=False)
            .agg(
                parsed_rows=('symbol', 'size'),
                parsed_long_quantity=('long_quantity', 'sum'),
                parsed_short_quantity=('short_quantity', 'sum'),
                parsed_average_price=('average_price', 'first'),
            )
        )
    audit = raw_audit.merge(parsed_audit, left_on='normalized_symbol', right_on='symbol', how='left')
    audit[['parsed_rows', 'parsed_long_quantity', 'parsed_short_quantity']] = audit[
        ['parsed_rows', 'parsed_long_quantity', 'parsed_short_quantity']
    ].fillna(0)
    audit['raw_rows'] = audit['raw_rows'].fillna(0)
    audit['row_match'] = audit['raw_rows'].astype(int) == audit['parsed_rows'].astype(int)
    audit['long_match'] = audit['raw_long_quantity'].round(6) == audit['parsed_long_quantity'].round(6)
    audit['short_match'] = audit['raw_short_quantity'].round(6) == audit['parsed_short_quantity'].round(6)
    audit['present_in_positions_df'] = audit[['row_match', 'long_match', 'short_match']].all(axis=1)
    raw_symbol_count = raw_option_df['normalized_symbol'].nunique()
    parsed_symbol_count = parsed_option_df['symbol'].nunique() if not parsed_option_df.empty else 0
    print('[Code Block 7] Option Leg Coverage Audit')
    print(f'- Accounts returned by Schwab: {len(acct_map):,}')
    print(f'- Selected account hash: {acct_hash}')
    print(f'- Raw option legs from Schwab: {len(raw_option_df):,}')
    print(f'- Parsed option legs in positions_df: {len(parsed_option_df):,}')
    print(f'- Distinct option symbols from Schwab: {raw_symbol_count:,}')
    print(f'- Distinct option symbols in positions_df: {parsed_symbol_count:,}')
    print(f'- Regex misses: {len(regex_miss_rows):,}')

    if audit['present_in_positions_df'].all():
        print('Status: PASS - every raw Schwab option leg is represented in positions_df.')
    else:
        print('Status: FAIL - some option legs are missing or quantities do not match.')
        display(
            audit.loc[
                ~audit['present_in_positions_df'],
                [
                    'underlying_symbol',
                    'normalized_symbol',
                    'expiration',
                    'put_call',
                    'raw_rows',
                    'parsed_rows',
                    'raw_long_quantity',
                    'parsed_long_quantity',
                    'raw_short_quantity',
                    'parsed_short_quantity',
                ],
            ].sort_values(['underlying_symbol', 'expiration', 'normalized_symbol'])
        )
    if regex_miss_rows:
        print('Fix suggestion: some option symbols failed the OCC regex, so keep those legs using instrument fields even when regex parsing fails.')
        display(pd.DataFrame(regex_miss_rows).sort_values(['underlying_symbol', 'raw_symbol']))

    if len(acct_map) > 1:
        print('Fix suggestion: your code currently pulls acct_map[0]. If your broker UI is showing another account, choose the matching hash or loop through every account.')


In [ ]:
# Code Block 9: DTE ladder
# =========================
# 9) Options expiration ladder
# =========================
from Quantapp.visualization.views.portfolio_profile.performance_structure import plot_options_expiration_ladder

fig = plot_options_expiration_ladder(positions_df)
if fig is not None:
    fig.show()


In [ ]:
# Code Block 10: Asset-level signed and unsigned analytics
# =========================
# 10) Asset-level signed and unsigned analytics
# =========================
rolling_windows = (21, 50, 200)
benchmark_label = 'SPY'

def _rolling_sharpe(values, window):
    rolling_mean = values.rolling(window).mean()
    rolling_std = values.rolling(window).std(ddof=0)
    sharpe = rolling_mean.div(rolling_std).mul(np.sqrt(252))
    return sharpe.mask(rolling_std.eq(0), 0.0).replace([np.inf, -np.inf], np.nan)

def _apply_zscore(values):
    if isinstance(values, pd.Series):
        return z_score(values)

    return values.apply(z_score)

def _latest_sorted_snapshot(frame):
    valid = frame.dropna(how='all')

    if valid.empty:
        return pd.Series(dtype=float)

    return valid.iloc[-1].sort_values(ascending=False)

def _log_summary(label, values):
    if isinstance(values, pd.DataFrame):
        valid = values.dropna(how='all')

        if valid.empty:
            return f'- {label}: empty DataFrame'

        return (
            f'- {label}: DataFrame {valid.shape[0]:,} x {valid.shape[1]} '
            f'({valid.index.min():%Y-%m-%d} to {valid.index.max():%Y-%m-%d})'
        )
    valid = values.dropna()

    if valid.empty:
        return f'- {label}: empty Series'

    return f'- {label}: Series {valid.shape[0]:,} rows ({valid.index.min():%Y-%m-%d} to {valid.index.max():%Y-%m-%d})'

# Daily return streams: unsigned raw returns and signed tradable returns
daily_returns = portfolio_closing_prices.pct_change().dropna(how='all')
benchmark_daily_returns = benchmark_close.pct_change().dropna()
signs = (
    net_direction['sign']
    .reindex(portfolio_closing_prices.columns)
    .fillna(1.0)
    .astype(float)
)
daily_returns_signed = daily_returns.mul(signs, axis=1)
benchmark_daily_returns_aligned = benchmark_daily_returns.reindex(daily_returns.index).dropna()
daily_returns_for_corr = daily_returns.reindex(benchmark_daily_returns_aligned.index)
daily_returns_signed_for_corr = daily_returns_signed.reindex(benchmark_daily_returns_aligned.index)
portfolio_daily_returns_for_corr = daily_returns_signed.mean(axis=1).reindex(benchmark_daily_returns_aligned.index)
print(f'[Code Block 9] Calculated daily return streams for {len(portfolio_closing_prices.columns)} assets.')

# Assets / Rolling horizon returns (unsigned + signed)
asset_return_windows = {
    window: portfolio_closing_prices.pct_change(window).dropna(how='all')
    for window in rolling_windows
}
asset_signed_return_windows = {
    window: frame.mul(signs, axis=1)
    for window, frame in asset_return_windows.items()
}
asset_return_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_return_windows.items()
}
asset_signed_return_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_signed_return_windows.items()
}
portfolio_assets_rolling_returns_21 = asset_return_windows[21]
portfolio_assets_rolling_returns_50 = asset_return_windows[50]
portfolio_assets_rolling_returns_200 = asset_return_windows[200]
portfolio_assets_rolling_signed_returns_21 = asset_signed_return_windows[21]
portfolio_assets_rolling_signed_returns_50 = asset_signed_return_windows[50]
portfolio_assets_rolling_signed_returns_200 = asset_signed_return_windows[200]
portfolio_assets_rolling_returns_z_scores_21 = asset_return_z_windows[21]
portfolio_assets_rolling_returns_z_scores_50 = asset_return_z_windows[50]
portfolio_assets_rolling_returns_z_scores_200 = asset_return_z_windows[200]
portfolio_assets_rolling_signed_return_z_scores_21 = asset_signed_return_z_windows[21]
portfolio_assets_rolling_signed_return_z_scores_50 = asset_signed_return_z_windows[50]
portfolio_assets_rolling_signed_return_z_scores_200 = asset_signed_return_z_windows[200]
print('[Code Block 9] Calculated asset rolling returns and signed return z-scores for 21/50/200-day windows.')

# Assets / Rolling Sharpe (unsigned + signed)
asset_sharpe_windows = {
    window: _rolling_sharpe(daily_returns, window)
    for window in rolling_windows
}
asset_signed_sharpe_windows = {
    window: _rolling_sharpe(daily_returns_signed, window)
    for window in rolling_windows
}
asset_sharpe_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_sharpe_windows.items()
}
asset_signed_sharpe_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_signed_sharpe_windows.items()
}
portfolio_assets_rolling_sharpe_21 = asset_sharpe_windows[21]
portfolio_assets_rolling_sharpe_50 = asset_sharpe_windows[50]
portfolio_assets_rolling_sharpe_200 = asset_sharpe_windows[200]
portfolio_assets_rolling_signed_sharpe_21 = asset_signed_sharpe_windows[21]
portfolio_assets_rolling_signed_sharpe_50 = asset_signed_sharpe_windows[50]
portfolio_assets_rolling_signed_sharpe_200 = asset_signed_sharpe_windows[200]
portfolio_assets_rolling_sharpe_z_scores_21 = asset_sharpe_z_windows[21]
portfolio_assets_rolling_sharpe_z_scores_50 = asset_sharpe_z_windows[50]
portfolio_assets_rolling_sharpe_z_scores_200 = asset_sharpe_z_windows[200]
portfolio_assets_rolling_signed_sharpe_z_scores_21 = asset_signed_sharpe_z_windows[21]
portfolio_assets_rolling_signed_sharpe_z_scores_50 = asset_signed_sharpe_z_windows[50]
portfolio_assets_rolling_signed_sharpe_z_scores_200 = asset_signed_sharpe_z_windows[200]
print('[Code Block 9] Calculated vectorized asset rolling Sharpe series for 21/50/200-day windows.')

# Assets / Rolling correlation to benchmark (unsigned + signed)
asset_corr_windows = {
    window: daily_returns_for_corr.rolling(window).corr(benchmark_daily_returns_aligned)
    for window in rolling_windows
}
asset_signed_corr_windows = {
    window: daily_returns_signed_for_corr.rolling(window).corr(benchmark_daily_returns_aligned)
    for window in rolling_windows
}
asset_corr_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_corr_windows.items()
}
asset_signed_corr_z_windows = {
    window: _apply_zscore(frame)
    for window, frame in asset_signed_corr_windows.items()
}
portfolio_assets_rolling_correlation_21 = asset_corr_windows[21]
portfolio_assets_rolling_correlation_50 = asset_corr_windows[50]
portfolio_assets_rolling_correlation_200 = asset_corr_windows[200]
portfolio_assets_rolling_signed_correlation_21 = asset_signed_corr_windows[21]
portfolio_assets_rolling_signed_correlation_50 = asset_signed_corr_windows[50]
portfolio_assets_rolling_signed_correlation_200 = asset_signed_corr_windows[200]
portfolio_assets_rolling_correlation_z_scores_21 = asset_corr_z_windows[21]
portfolio_assets_rolling_correlation_z_scores_50 = asset_corr_z_windows[50]
portfolio_assets_rolling_correlation_z_scores_200 = asset_corr_z_windows[200]
portfolio_assets_rolling_signed_correlation_z_scores_21 = asset_signed_corr_z_windows[21]
portfolio_assets_rolling_signed_correlation_z_scores_50 = asset_signed_corr_z_windows[50]
portfolio_assets_rolling_signed_correlation_z_scores_200 = asset_signed_corr_z_windows[200]
print(f'[Code Block 9] Calculated asset rolling correlations to {benchmark_label} for 21/50/200-day windows.')

# Assets / Latest z-scores by metric
latest_assets_return_z_scores_21 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_return_z_scores_21)
latest_assets_return_z_scores_50 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_return_z_scores_50)
latest_assets_return_z_scores_200 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_return_z_scores_200)
latest_assets_sharpe_z_scores_21 = _latest_sorted_snapshot(portfolio_assets_rolling_sharpe_z_scores_21)
latest_assets_sharpe_z_scores_50 = _latest_sorted_snapshot(portfolio_assets_rolling_sharpe_z_scores_50)
latest_assets_sharpe_z_scores_200 = _latest_sorted_snapshot(portfolio_assets_rolling_sharpe_z_scores_200)
latest_assets_signed_sharpe_z_scores_21 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_sharpe_z_scores_21)
latest_assets_signed_sharpe_z_scores_50 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_sharpe_z_scores_50)
latest_assets_signed_sharpe_z_scores_200 = _latest_sorted_snapshot(portfolio_assets_rolling_signed_sharpe_z_scores_200)

# Display labels: add parentheses only for signed series with negative direction
def format_ticker(ticker, sign):
    return f'({ticker})' if sign < 0 else ticker

latest_assets_return_z_scores_21.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_return_z_scores_21.index]
latest_assets_return_z_scores_50.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_return_z_scores_50.index]
latest_assets_return_z_scores_200.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_return_z_scores_200.index]
latest_assets_sharpe_z_scores_21.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_sharpe_z_scores_21.index]
latest_assets_sharpe_z_scores_50.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_sharpe_z_scores_50.index]
latest_assets_sharpe_z_scores_200.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_sharpe_z_scores_200.index]
latest_assets_signed_sharpe_z_scores_21.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_signed_sharpe_z_scores_21.index]
latest_assets_signed_sharpe_z_scores_50.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_signed_sharpe_z_scores_50.index]
latest_assets_signed_sharpe_z_scores_200.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_signed_sharpe_z_scores_200.index]

# Portfolio aggregates (signed equal-weight daily rebalance)
portfolio_daily_returns = daily_returns_signed.mean(axis=1)
portfolio_equity = (1 + portfolio_daily_returns).cumprod()
portfolio_return_21 = portfolio_equity.pct_change(21).dropna()
portfolio_return_50 = portfolio_equity.pct_change(50).dropna()
portfolio_return_200 = portfolio_equity.pct_change(200).dropna()
portfolio_rolling_sharpe_21 = _rolling_sharpe(portfolio_daily_returns, 21).dropna()
portfolio_rolling_sharpe_50 = _rolling_sharpe(portfolio_daily_returns, 50).dropna()
portfolio_rolling_sharpe_200 = _rolling_sharpe(portfolio_daily_returns, 200).dropna()

# Benchmark analytics (unsigned benchmark series)
benchmark_equity = (1 + benchmark_daily_returns).cumprod()
benchmark_rolling_sharpe_21 = _rolling_sharpe(benchmark_daily_returns, 21).dropna()
benchmark_rolling_sharpe_50 = _rolling_sharpe(benchmark_daily_returns, 50).dropna()
benchmark_rolling_sharpe_200 = _rolling_sharpe(benchmark_daily_returns, 200).dropna()
portfolio_rolling_signed_correlation_21 = portfolio_daily_returns_for_corr.rolling(21).corr(benchmark_daily_returns_aligned).dropna()
portfolio_rolling_signed_correlation_50 = portfolio_daily_returns_for_corr.rolling(50).corr(benchmark_daily_returns_aligned).dropna()
portfolio_rolling_signed_correlation_200 = portfolio_daily_returns_for_corr.rolling(200).corr(benchmark_daily_returns_aligned).dropna()
print(f'[Code Block 9] Calculated portfolio and benchmark aggregate analytics versus {benchmark_label}.')
calculation_log = [
    '[Code Block 9] Summary:',
    f'- Assets analyzed: {len(portfolio_closing_prices.columns)}',
    f'- Rolling windows: {", ".join(str(window) for window in rolling_windows)} trading days',
    _log_summary('Asset daily returns', daily_returns),
    _log_summary('Signed asset daily returns', daily_returns_signed),
    _log_summary('Signed asset rolling returns (200d)', portfolio_assets_rolling_signed_returns_200),
    _log_summary('Asset rolling Sharpe (200d)', portfolio_assets_rolling_sharpe_200),
    _log_summary(f'Asset rolling correlation to {benchmark_label} (200d)', portfolio_assets_rolling_signed_correlation_200),
    _log_summary('Portfolio equity', portfolio_equity),
    _log_summary('Portfolio rolling Sharpe (200d)', portfolio_rolling_sharpe_200),
    _log_summary(f'Portfolio rolling correlation to {benchmark_label} (200d)', portfolio_rolling_signed_correlation_200),
]
print("\n".join(calculation_log))


In [ ]:
# Code Block 11: Plots
# =========================
# 11) Plots
# =========================
from Quantapp.visualization.views.portfolio_profile.performance_structure import (
    plot_equity_curve,
    plot_rolling_correlation,
    plot_rolling_sharpe_zscore,
)

portfolio_equity_fig = plot_equity_curve(
    portfolio_equity,
    benchmark_equity,
    benchmark_label=benchmark_str,
)
portfolio_equity_fig.show()

rolling_sharpe_fig = plot_rolling_sharpe_zscore(
    {
        21: portfolio_rolling_sharpe_21,
        50: portfolio_rolling_sharpe_50,
        200: portfolio_rolling_sharpe_200,
    },
    {
        21: benchmark_rolling_sharpe_21,
        50: benchmark_rolling_sharpe_50,
        200: benchmark_rolling_sharpe_200,
    },
    benchmark_label=benchmark_str,
    default_window=200,
)
rolling_sharpe_fig.show()

rolling_correlation_fig = plot_rolling_correlation(
    {
        21: portfolio_rolling_signed_correlation_21,
        50: portfolio_rolling_signed_correlation_50,
        200: portfolio_rolling_signed_correlation_200,
    },
    benchmark_label=benchmark_str,
)
rolling_correlation_fig.show()


In [ ]:
# Code Block 12: Relative price strength z-score plots
# Relative Price Strength Z-Score Plots
from Quantapp.analytics import TimeSeriesAnalytics as Rolling, RiskRelativeAnalytics
from Quantapp.visualization.views.portfolio_profile.performance_structure import (
    format_snapshot_map,
    plot_benchmark_snapshot_zscores,
    plot_z_score_diff_dropdown,
)

rolling = Rolling()
risk_relative_analytics = RiskRelativeAnalytics()
time_frame_map = {
    '21': time_frame_short,
    '50': time_frame_mid,
    '200': time_frame_long,
}
sign_series = net_direction['sign'].reindex(portfolio_closing_prices.columns).fillna(1.0).astype(float)

benchmark_snapshot = risk_relative_analytics.build_multi_asset_benchmark_snapshot(
    analytics=rolling,
    asset_close=portfolio_closing_prices,
    benchmark_close=benchmark_close,
    time_frame_map=time_frame_map,
    sign_map=sign_series,
)
windows_signed = format_snapshot_map(
    benchmark_snapshot['signed_asset_latest_zscores'],
    sign_series,
)
windows_unsigned = benchmark_snapshot['unsigned_asset_latest_zscores']
windows_benchmark_minus_assets_signed = format_snapshot_map(
    benchmark_snapshot['signed_spread_latest_zscores'],
    sign_series,
)
windows_benchmark_minus_assets_unsigned = benchmark_snapshot['unsigned_spread_latest_zscores']
signed_return_zscore_map = format_snapshot_map(
    benchmark_snapshot['signed_return_latest_zscores'],
    sign_series,
)

snapshot_fig = plot_benchmark_snapshot_zscores(
    windows_signed=windows_signed,
    windows_unsigned=windows_unsigned,
    windows_benchmark_minus_assets_signed=windows_benchmark_minus_assets_signed,
    windows_benchmark_minus_assets_unsigned=windows_benchmark_minus_assets_unsigned,
    sign_series=sign_series,
    benchmark_label=benchmark_str,
)
snapshot_fig.show()

return_zscore_diff_fig = plot_z_score_diff_dropdown(
    signed_return_zscore_map,
    title_metric='Signed Return',
)
return_zscore_diff_fig.show()


In [ ]:
# Code Block 13: Portfolio allocation optimization helpers
# This tells you to increase capital allocation to the asset. This does not tell you whether it is a good investment.
from scipy.optimize import minimize


def optimal_allocation(dataframe, risk_free_rate=0.0):
    """Calculate max-Sharpe long-only portfolio weights for a price DataFrame."""
    returns_df = dataframe.pct_change().dropna()
    mean_returns = returns_df.mean() * 252
    cov_matrix = returns_df.cov() * 252
    num_assets = len(dataframe.columns)

    def neg_sharpe(weights):
        ret = np.dot(weights, mean_returns)
        vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        return -(ret - risk_free_rate) / vol

    constraints = ({'type': 'eq', 'fun': lambda weights: np.sum(weights) - 1})
    bounds = tuple((0.0, 1.0) for _ in range(num_assets))
    init_guess = [1.0 / num_assets] * num_assets
    result = minimize(
        neg_sharpe,
        init_guess,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints,
    )
    weights_df = pd.DataFrame(result.x, index=dataframe.columns, columns=['weight'])
    return weights_df.round(2)


def rolling_optimal_allocation(dataframe, window=50, risk_free_rate=0.0):
    """Apply max-Sharpe allocation over rolling windows of a price DataFrame."""
    returns_df = dataframe.pct_change().dropna()
    rolling_weights_list = []

    for i in range(window, len(returns_df) + 1):
        slice_data = dataframe.iloc[i - window : i]
        weights_df = optimal_allocation(slice_data, risk_free_rate=risk_free_rate)
        rolling_weights_list.append(weights_df['weight'].values)

    rolling_index = returns_df.index[window - 1 :]
    return pd.DataFrame(
        rolling_weights_list,
        index=rolling_index,
        columns=dataframe.columns,
    )


benchmark = qa_yf.download('SPY', period=period, interval=interval, auto_adjust=True, progress=False)['Close']
raw_prices = portfolio_closing_prices.copy()

for ticker in raw_prices.columns:
    if ticker in net_direction.index:
        direction = net_direction.loc[ticker, 'directional_value']

        if direction < 0:
            first_price = raw_prices[ticker].iloc[0]
            flipped = (first_price ** 2) / raw_prices[ticker]
            flipped = flipped.astype(float)
            flipped.replace([np.inf, -np.inf], np.nan, inplace=True)
            flipped.fillna(method='ffill', inplace=True)
            flipped.fillna(method='bfill', inplace=True)
            raw_prices[ticker] = flipped

raw_prices = raw_prices.drop(columns=['VXX'], errors='ignore')


In [ ]:
# Code Block 14: Rolling allocations
# rolling allocations
from Quantapp.visualization.views.portfolio_profile.performance_structure import plot_rolling_portfolio_allocation

rolling_optimal_allocations = rolling_optimal_allocation(raw_prices, window=selected_time_frame)
rolling_allocation_fig = plot_rolling_portfolio_allocation(rolling_optimal_allocations)
if rolling_allocation_fig is not None:
    rolling_allocation_fig.show()


In [ ]:
# Code Block 15: Archived rolling sharpe snippet
# Archived plotting code was moved into Quantapp.visualization.views.portfolio_profile.performance_structure.
# Use plot_rolling_sharpe_zscore for the active rolling Sharpe view.
